# 29. Convolutional neural networks and image features

**Level:** Intermediate

**Prerequisites:** 27-28: tensors and neural training.

[Course index](00_START_HERE.ipynb) | [Previous](28_Training_a_PyTorch_MLP.ipynb) | [Next](30_RNNs_Sequences_and_Seq2Seq.ipynb)

**Topics:** CNN, Convolution, Channels, Filters, Pooling, Images.

**Matching script:** `examples\29_Convolutional_Neural_Networks.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

A convolution applies a shared kernel across local input regions. Shared weights reduce the
parameter count compared with a fully connected layer over every pixel. Different channels
encode different learned features, and pooling or strides can reduce spatial resolution.

PyTorch Conv2d normally receives (batch, channels, height, width). Confusing channels with a
spatial dimension can produce wrong behavior even when shapes happen to be legal. Pixel
normalization must match training, and data augmentation should preserve the target meaning.

Convolutional feature maps can respond to local edges and patterns; deeper learned layers can
combine them. A manually chosen edge kernel below illustrates convolution, not a trained
image classifier. Real vision evaluation needs representative held-out images, labels,
deployment constraints, and checks for shortcut features.

## Mathematics or data contract

A valid 3x3 convolution over a 5x5 input with stride 1 and no padding produces a 3x3 feature map.

## Important limitation

The exercise uses a fixed kernel, so there is no learned visual recognition or accuracy claim.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Inspect a convolution's geometry and parameter sharing

**Purpose:** Apply one known kernel to a synthetic image.

### Read the code in this order
1. Construct a 1x1x5x5 tensor with a vertical stripe.
2. Install a hand-written 3x3 kernel without gradient updates.
3. Run convolution and pooling.
4. Assert output shapes and nonzero response.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- `with`/`async with` bounds a resource lifetime or a temporary execution mode.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
feature shape: (1, 1, 3, 3) pooled response: 3.0
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import torch
from torch import nn

torch.set_num_threads(1)
image = torch.zeros(1, 1, 5, 5)
image[:, :, :, 2] = 1
convolution = nn.Conv2d(1, 1, kernel_size=3, bias=False)
kernel = torch.tensor([[-1., 0., 1.], [-1., 0., 1.], [-1., 0., 1.]])
with torch.no_grad():
    convolution.weight.copy_(kernel.reshape(1, 1, 3, 3))
features = convolution(image)
pooled = nn.MaxPool2d(2)(features.abs())
assert features.shape == (1, 1, 3, 3) and pooled.shape == (1, 1, 1, 1)
assert torch.count_nonzero(features).item() > 0
assert sum(parameter.numel() for parameter in convolution.parameters()) == 9
print("feature shape:", tuple(features.shape), "pooled response:", pooled.item())

feature shape: (1, 1, 3, 3) pooled response: 3.0


### Interpret example 1

The result should demonstrate this purpose: **Apply one known kernel to a synthetic image.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Worked example 2: Train a CNN on synthetic stripe images

**Purpose:** Move from a fixed kernel to actual learned visual classification on a held-out set.

### Read the code in this order
1. Generate noisy horizontal and vertical stripe examples locally.
2. Split before training.
3. Train a small CNN on CPU.
4. Assert reduced training loss and held-out accuracy.

### Syntax and behavior to notice
- `backward()` computes/accumulates gradients through the recorded computation graph.
- `zero_grad()` clears earlier gradients before the next independent training update.
- `optimizer.step()` updates only the parameters owned by that optimizer.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.
- `with`/`async with` bounds a resource lifetime or a temporary execution mode.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
synthetic stripe held-out accuracy: 1.0
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [2]:
import numpy as np
import torch
from torch import nn
from sklearn.model_selection import train_test_split

torch.set_num_threads(1)
torch.manual_seed(42)
rng = np.random.default_rng(42)
images = rng.normal(0, 0.08, (160, 1, 8, 8)).astype(np.float32)
labels = np.arange(160) % 2
for index, label in enumerate(labels):
    position = int(rng.integers(2, 6))
    if label == 0:
        images[index, 0, :, position] += 1
    else:
        images[index, 0, position, :] += 1
train_ids, test_ids = train_test_split(np.arange(160), stratify=labels, random_state=42)
train_X, test_X = torch.tensor(images[train_ids]), torch.tensor(images[test_ids])
train_y = torch.tensor(labels[train_ids], dtype=torch.long)
model = nn.Sequential(nn.Conv2d(1, 4, 3), nn.ReLU(), nn.Flatten(), nn.Linear(4*6*6, 2))
optimizer, criterion = torch.optim.Adam(model.parameters(), lr=0.02), nn.CrossEntropyLoss()
initial = criterion(model(train_X), train_y).item()
for _ in range(65):
    optimizer.zero_grad()
    loss = criterion(model(train_X), train_y)
    loss.backward()
    optimizer.step()
model.eval()
with torch.inference_mode():
    accuracy = float((model(test_X).argmax(1).numpy() == labels[test_ids]).mean())
assert loss.item() < initial and accuracy > 0.9
print("synthetic stripe held-out accuracy:", round(accuracy, 3))

synthetic stripe held-out accuracy: 1.0


### Interpret example 2

The result should demonstrate this purpose: **Move from a fixed kernel to actual learned visual classification on a held-out set.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Train a small classifier on synthetic horizontal/vertical stripes and test on unseen noisy stripes.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `CNN` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: A valid 3x3 convolution over a 5x5 input with stride 1 and no padding produces a 3x3 feature map.
- Name the boundary of the demonstration: The exercise uses a fixed kernel, so there is no learned visual recognition or accuracy claim.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://docs.pytorch.org/tutorials/beginner/basics/intro.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.